# Producto U1 — Jhan Logan Ramos Quispe

**Dimensión:** Duración del flujo de red (`flow_duration`) — U1 batch, regresión

**Rol en el equipo:** Streaming / Kafka

**Curso:** Big Data · lambda26 · Proyecto Sello (equipo LLSW3, sección GU)

**Caso:** análisis de tráfico de red del campus universitario (dataset propio, ~400 000 flujos, capturado vía Suricata)

**Metodología:** CRISP-DM — Fases 1 a 5 (hasta modelado/evaluación; el despliegue es Unidad 2)


## Arquitectura Big Data (contexto — no es una fase de CRISP-DM)

Este notebook implementa la **ruta batch** de la arquitectura **Lambda** declarada en el
[Brief técnico-analítico](../../docs/proyecto-sello/brief.md) (Hito S2): capa batch (este
notebook) + capa de velocidad (Kafka, contenido de Unidad 2). El detalle completo de la
decisión Lambda vs. Kappa está en el brief.

## Fase 1 — Comprensión del negocio (CRISP-DM)

**Pregunta de negocio (dimensión propia):** ¿Qué duración histórica (`flow_duration`) han
tenido los flujos según sus características iniciales, y qué duración se puede esperar?

**Objetivo de minería de datos:** entrenar un modelo de **regresión** que prediga
`flow_duration` a partir de variables disponibles temprano en el ciclo de vida del flujo.

**Decisión que habilita:** dimensionar ventanas de sesión y tiempos de retención en Kafka
(Unidad 2) según la duración esperada.

**Criterio de éxito:** el modelo debe superar con margen claro una línea base ingenua
(predecir siempre la mediana histórica de `flow_duration`) en RMSE (referencia orientativa:
reducción de al menos 20% de RMSE frente a la línea base).


## Fase 2 — Comprensión de los datos (CRISP-DM)

### Extracción con esquema explícito


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, DoubleType
)
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("u1-producto-jhan-duracion")
    .getOrCreate()
)

RUTA_DATOS = "/opt/data/TRCU.csv"

# Hallazgo documentado en S03: con header=True + StructType explícito, Spark asigna
# los campos por POSICION, no por nombre de columna. Si el orden de StructField no
# coincide exactamente con el orden físico del CSV, los valores se corrompen en
# silencio (sin lanzar error). Por eso el orden de abajo respeta el header real
# observado en el dataset, columna por columna.

esquema_flujos = StructType([
    StructField("flow_id", StringType(), True),
    StructField("src_addr", StringType(), True),
    StructField("src_port", IntegerType(), True),
    StructField("dst_addr", StringType(), True),
    StructField("dst_port", IntegerType(), True),
    StructField("ip_prot", IntegerType(), True),
    StructField("timestamp", LongType(), True),
    StructField("flow_duration", DoubleType(), True),
    StructField("down_up_ratio", DoubleType(), True),
    StructField("pkt_len_max", DoubleType(), True),
    StructField("pkt_len_min", DoubleType(), True),
    StructField("pkt_len_mean", DoubleType(), True),
    StructField("pkt_len_var", DoubleType(), True),
    StructField("pkt_len_std", DoubleType(), True),
    StructField("bytes_per_s", DoubleType(), True),
    StructField("pkt_per_s", DoubleType(), True),
    StructField("fwd_pkt_per_s", DoubleType(), True),
    StructField("bwd_pkt_per_s", DoubleType(), True),
    StructField("fwd_pkt_cnt", IntegerType(), True),
    StructField("fwd_pkt_len_tot", DoubleType(), True),
    StructField("fwd_pkt_len_max", DoubleType(), True),
    StructField("fwd_pkt_len_min", DoubleType(), True),
    StructField("fwd_pkt_len_mean", DoubleType(), True),
    StructField("fwd_pkt_len_std", DoubleType(), True),
    StructField("fwd_pkt_hdr_len_tot", IntegerType(), True),
    StructField("fwd_pkt_hdr_len_min", IntegerType(), True),
    StructField("fwd_non_empty_pkt_cnt", IntegerType(), True),
    StructField("bwd_pkt_cnt", IntegerType(), True),
    StructField("bwd_pkt_len_tot", DoubleType(), True),
    StructField("bwd_pkt_len_max", DoubleType(), True),
    StructField("bwd_pkt_len_min", DoubleType(), True),
    StructField("bwd_pkt_len_mean", DoubleType(), True),
    StructField("bwd_pkt_len_std", DoubleType(), True),
    StructField("bwd_pkt_hdr_len_tot", IntegerType(), True),
    StructField("bwd_pkt_hdr_len_min", IntegerType(), True),
    StructField("bwd_non_empty_pkt_cnt", IntegerType(), True),
    StructField("iat_max", DoubleType(), True),
    StructField("iat_min", DoubleType(), True),
    StructField("iat_mean", DoubleType(), True),
    StructField("iat_std", DoubleType(), True),
    StructField("fwd_iat_tot", DoubleType(), True),
    StructField("fwd_iat_max", DoubleType(), True),
    StructField("fwd_iat_min", DoubleType(), True),
    StructField("fwd_iat_mean", DoubleType(), True),
    StructField("fwd_iat_std", DoubleType(), True),
    StructField("bwd_iat_tot", DoubleType(), True),
    StructField("bwd_iat_max", DoubleType(), True),
    StructField("bwd_iat_min", DoubleType(), True),
    StructField("bwd_iat_mean", DoubleType(), True),
    StructField("bwd_iat_std", DoubleType(), True),
    StructField("active_max", DoubleType(), True),
    StructField("active_min", DoubleType(), True),
    StructField("active_mean", DoubleType(), True),
    StructField("active_std", DoubleType(), True),
    StructField("idle_max", DoubleType(), True),
    StructField("idle_min", DoubleType(), True),
    StructField("idle_mean", DoubleType(), True),
    StructField("idle_std", DoubleType(), True),
    StructField("flag_SYN", IntegerType(), True),
    StructField("flag_fin", IntegerType(), True),
    StructField("flag_rst", IntegerType(), True),
    StructField("flag_ack", IntegerType(), True),
    StructField("flag_psh", IntegerType(), True),
    StructField("fwd_flag_psh", IntegerType(), True),
    StructField("bwd_flag_psh", IntegerType(), True),
    StructField("flag_urg", IntegerType(), True),
    StructField("fwd_flag_urg", IntegerType(), True),
    StructField("bwd_flag_urg", IntegerType(), True),
    StructField("flag_cwr", IntegerType(), True),
    StructField("flag_ece", IntegerType(), True),
    StructField("fwd_bulk_bytes_mean", DoubleType(), True),
    StructField("fwd_bulk_pkt_mean", DoubleType(), True),
    StructField("fwd_bulk_rate_mean", DoubleType(), True),
    StructField("bwd_bulk_bytes_mean", DoubleType(), True),
    StructField("bwd_bulk_pkt_mean", DoubleType(), True),
    StructField("bwd_bulk_rate_mean", DoubleType(), True),
    StructField("fwd_subflow_bytes_mean", DoubleType(), True),
    StructField("fwd_subflow_pkt_mean", DoubleType(), True),
    StructField("bwd_subflow_bytes_mean", DoubleType(), True),
    StructField("bwd_subflow_pkt_mean", DoubleType(), True),
    StructField("fwd_tcp_init_win_bytes", IntegerType(), True),
    StructField("bwd_tcp_init_win_bytes", IntegerType(), True),
    StructField("label", StringType(), True),
])

df = spark.read.csv(RUTA_DATOS, header=True, schema=esquema_flujos)

with open(RUTA_DATOS, "r", encoding="utf-8") as f:
    cabecera_real = f.readline().strip().split(",")
assert cabecera_real == [c.name for c in esquema_flujos.fields], (
    "El orden del esquema no coincide con el header real del CSV — revisar antes de continuar."
)

df.printSchema()
df.show(5, truncate=False)
print("Filas totales:", df.count())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 04:13:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- flow_id: string (nullable = true)
 |-- src_addr: string (nullable = true)
 |-- src_port: integer (nullable = true)
 |-- dst_addr: string (nullable = true)
 |-- dst_port: integer (nullable = true)
 |-- ip_prot: integer (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- flow_duration: double (nullable = true)
 |-- down_up_ratio: double (nullable = true)
 |-- pkt_len_max: double (nullable = true)
 |-- pkt_len_min: double (nullable = true)
 |-- pkt_len_mean: double (nullable = true)
 |-- pkt_len_var: double (nullable = true)
 |-- pkt_len_std: double (nullable = true)
 |-- bytes_per_s: double (nullable = true)
 |-- pkt_per_s: double (nullable = true)
 |-- fwd_pkt_per_s: double (nullable = true)
 |-- bwd_pkt_per_s: double (nullable = true)
 |-- fwd_pkt_cnt: integer (nullable = true)
 |-- fwd_pkt_len_tot: double (nullable = true)
 |-- fwd_pkt_len_max: double (nullable = true)
 |-- fwd_pkt_len_min: double (nullable = true)
 |-- fwd_pkt_len_mean: double (nullable = true)
 |

26/09/11 04:13:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------------------------------+--------------+--------+--------------+--------+-------+----------------+-------------+-------------+-----------+-----------+------------+-------------+-----------+------------+---------+-------------+-------------+-----------+---------------+---------------+---------------+----------------+---------------+-------------------+-------------------+---------------------+-----------+---------------+---------------+---------------+----------------+---------------+-------------------+-------------------+---------------------+--------+-------+------------+-------------+-----------+-----------+-----------+------------+-------------+-----------+-----------+-----------+------------+-------------+----------+----------+-----------+----------+--------+--------+---------+--------+--------+--------+--------+--------+--------+------------+------------+--------+------------+------------+--------+--------+-------------------+-----------------+------------------

[Stage 1:==>                                                      (1 + 23) / 24]

Filas totales: 397354


### Exploración inicial (EDA)


In [2]:
print("Resumen estadistico de flow_duration por protocolo:")
df.select("flow_duration", "iat_mean", "fwd_iat_mean", "bwd_iat_mean").describe().show()

print("Nulos por columna clave:")
for columna in ["flow_duration", "iat_mean", "ip_prot"]:
    n_nulos = df.filter(F.col(columna).isNull()).count()
    print(f"  {columna}: {n_nulos} nulos")

percentiles = df.approxQuantile("flow_duration", [0.0, 0.25, 0.5, 0.75, 1.0], 0.01)
print("flow_duration -> min/p25/mediana/p75/max:", percentiles)

linea_base_ingenua = percentiles[2]  # mediana
print("Linea base ingenua (mediana historica de flow_duration):", linea_base_ingenua)

print("Distribucion por protocolo (ip_prot):")
df.groupBy("ip_prot").count().orderBy(F.desc("count")).show()


Resumen estadistico de flow_duration por protocolo:


+-------+--------------------+--------------------+--------------------+-----------------+
|summary|       flow_duration|            iat_mean|        fwd_iat_mean|     bwd_iat_mean|
+-------+--------------------+--------------------+--------------------+-----------------+
|  count|              397354|              397354|              397354|           397354|
|   mean|1.0620946267620308E7|   3963328.166860908|  3086354.2544786576|880064.8563923732|
| stddev| 3.026542849501702E7|1.3829478112778643E7|1.2357251364623569E7|6633854.375359517|
|    min|                 0.0|                 0.0|                 0.0|              0.0|
|    max|        1.19999999E8|        1.19999996E8|        1.19999996E8|     1.19999947E8|
+-------+--------------------+--------------------+--------------------+-----------------+

Nulos por columna clave:


  flow_duration: 0 nulos


  iat_mean: 0 nulos


  ip_prot: 0 nulos


flow_duration -> min/p25/mediana/p75/max: [0.0, 0.0, 0.0, 1.0, 119999999.0]
Linea base ingenua (mediana historica de flow_duration): 0.0
Distribucion por protocolo (ip_prot):


+-------+------+
|ip_prot| count|
+-------+------+
|     17|374483|
|      6| 22579|
|      2|   243|
|      1|    49|
+-------+------+



## Fase 3 — Preparación de los datos (CRISP-DM)

### Transformación y agregación


In [3]:
df_jhan = (
    df
    .withColumn(
        "protocolo",
        F.when(F.col("ip_prot") == 6, F.lit("TCP"))
         .when(F.col("ip_prot") == 17, F.lit("UDP"))
         .otherwise(F.lit("OTRO"))
    )
    .filter(F.col("flow_duration").isNotNull() & (F.col("flow_duration") > 0))
)

df_jhan.explain(True)  # confirmar en que punto Spark deja de ser perezoso

resumen_duracion = (
    df_jhan.groupBy("protocolo")
    .agg(
        F.avg("flow_duration").alias("duracion_prom_us"),
        F.expr("percentile_approx(flow_duration, 0.5)").alias("duracion_mediana_us"),
        F.count("*").alias("n_flujos"),
    )
)
resumen_duracion.show()


== Parsed Logical Plan ==
'Filter 'and('isNotNull('flow_duration), '`>`('flow_duration, 0))
+- Project [flow_id#0, src_addr#1, src_port#2, dst_addr#3, dst_port#4, ip_prot#5, timestamp#6L, flow_duration#7, down_up_ratio#8, pkt_len_max#9, pkt_len_min#10, pkt_len_mean#11, pkt_len_var#12, pkt_len_std#13, bytes_per_s#14, pkt_per_s#15, fwd_pkt_per_s#16, bwd_pkt_per_s#17, fwd_pkt_cnt#18, fwd_pkt_len_tot#19, fwd_pkt_len_max#20, fwd_pkt_len_min#21, fwd_pkt_len_mean#22, fwd_pkt_len_std#23, fwd_pkt_hdr_len_tot#24, ... 59 more fields]
   +- Relation [flow_id#0,src_addr#1,src_port#2,dst_addr#3,dst_port#4,ip_prot#5,timestamp#6L,flow_duration#7,down_up_ratio#8,pkt_len_max#9,pkt_len_min#10,pkt_len_mean#11,pkt_len_var#12,pkt_len_std#13,bytes_per_s#14,pkt_per_s#15,fwd_pkt_per_s#16,bwd_pkt_per_s#17,fwd_pkt_cnt#18,fwd_pkt_len_tot#19,fwd_pkt_len_max#20,fwd_pkt_len_min#21,fwd_pkt_len_mean#22,fwd_pkt_len_std#23,fwd_pkt_hdr_len_tot#24,... 58 more fields] csv

== Analyzed Logical Plan ==
flow_id: string, src_a

+---------+--------------------+-------------------+--------+
|protocolo|    duracion_prom_us|duracion_mediana_us|n_flujos|
+---------+--------------------+-------------------+--------+
|     OTRO|1.2868307241666667E7|          3501615.0|     240|
|      UDP|4.3848324977936536E7|          9211691.0|   87203|
|      TCP|2.5283146336503245E7|           228220.0|   15563|
+---------+--------------------+-------------------+--------+



### Calidad de datos y particionamiento analítico


In [4]:
df_dedup = df_jhan.dropDuplicates(["flow_id"])

df_limpio = df_dedup.na.fill({
    "flow_duration": 0.0,
    "iat_mean": 0.0,
    "fwd_iat_mean": 0.0,
    "bwd_iat_mean": 0.0,
    "fwd_pkt_cnt": 0,
    "bwd_pkt_cnt": 0,
})

RUTA_SALIDA = "/opt/artifacts/jhan/flujos_particionado"
(
    df_limpio.write.mode("overwrite")
    .partitionBy("protocolo")
    .parquet(RUTA_SALIDA)
)

df_verificacion = spark.read.parquet(RUTA_SALIDA)
print("Filas tras deduplicacion:", df_limpio.count())
print("Filas leidas de vuelta desde Parquet:", df_verificacion.count())

df_verificacion.filter(F.col("protocolo") == "TCP").explain(True)  # confirmar PartitionFilters

26/09/11 02:36:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/11 02:36:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:36:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers


26/09/11 02:36:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:36:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


26/09/11 02:36:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/11 02:36:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:36:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/11 02:36:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:36:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


26/09/11 02:36:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/11 02:36:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:36:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Filas tras deduplicacion: 39728


Filas leidas de vuelta desde Parquet: 39728
== Parsed Logical Plan ==
'Filter '`=`('protocolo, TCP)
+- Relation [flow_id#2245,src_addr#2246,src_port#2247,dst_addr#2248,dst_port#2249,ip_prot#2250,timestamp#2251L,flow_duration#2252,down_up_ratio#2253,pkt_len_max#2254,pkt_len_min#2255,pkt_len_mean#2256,pkt_len_var#2257,pkt_len_std#2258,bytes_per_s#2259,pkt_per_s#2260,fwd_pkt_per_s#2261,bwd_pkt_per_s#2262,fwd_pkt_cnt#2263,fwd_pkt_len_tot#2264,fwd_pkt_len_max#2265,fwd_pkt_len_min#2266,fwd_pkt_len_mean#2267,fwd_pkt_len_std#2268,fwd_pkt_hdr_len_tot#2269,... 59 more fields] parquet

== Analyzed Logical Plan ==
flow_id: string, src_addr: string, src_port: int, dst_addr: string, dst_port: int, ip_prot: int, timestamp: bigint, flow_duration: double, down_up_ratio: double, pkt_len_max: double, pkt_len_min: double, pkt_len_mean: double, pkt_len_var: double, pkt_len_std: double, bytes_per_s: double, pkt_per_s: double, fwd_pkt_per_s: double, bwd_pkt_per_s: double, fwd_pkt_cnt: int, fwd_pkt_len_tot: d

### Selección de predictores y ensamblado del vector de features


In [5]:
from pyspark.ml.feature import VectorAssembler

predictores_jhan = [
    "ip_prot", "iat_mean", "iat_std", "fwd_iat_mean", "bwd_iat_mean",
    "fwd_pkt_cnt", "bwd_pkt_cnt", "pkt_len_mean", "flag_SYN", "flag_ack",
    "active_mean", "idle_mean", "fwd_tcp_init_win_bytes", "bwd_tcp_init_win_bytes",
]

ensamblador = VectorAssembler(inputCols=predictores_jhan, outputCol="features", handleInvalid="skip")
dataset_ml = (
    ensamblador.transform(df_limpio)
    .select("features", F.col("flow_duration").alias("valor_real"))
)

df_train, df_test = dataset_ml.randomSplit([0.8, 0.2], seed=42)
print("Filas de entrenamiento:", df_train.count(), " / Filas de prueba:", df_test.count())


Filas de entrenamiento: 31974  / Filas de prueba: 7754


## Fase 4 — Modelado (CRISP-DM)


In [6]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor

configuraciones_jhan = {
    "LinearRegression base": LinearRegression(featuresCol="features", labelCol="valor_real"),
    "LinearRegression + regularizacion": LinearRegression(featuresCol="features", labelCol="valor_real", regParam=0.1, elasticNetParam=0.5),
    "RandomForestRegressor": RandomForestRegressor(featuresCol="features", labelCol="valor_real", seed=42),
}

modelos_entrenados_jhan = {}
predicciones_jhan = {}
for nombre, estimador in configuraciones_jhan.items():
    modelo = estimador.fit(df_train)
    modelos_entrenados_jhan[nombre] = modelo
    predicciones_jhan[nombre] = modelo.transform(df_test)
    print("Entrenado:", nombre)


26/09/11 02:36:13 WARN Instrumentation: [f7d6f1d2] regParam is zero, which might cause numerical instability and overfitting.


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


Entrenado: LinearRegression base


Entrenado: LinearRegression + regularizacion


Entrenado: RandomForestRegressor


## Fase 5 — Evaluación (CRISP-DM)


In [7]:
from pyspark.ml.evaluation import RegressionEvaluator

ev_rmse = RegressionEvaluator(labelCol="valor_real", metricName="rmse")
ev_r2 = RegressionEvaluator(labelCol="valor_real", metricName="r2")
ev_mae = RegressionEvaluator(labelCol="valor_real", metricName="mae")

pred_baseline = df_test.withColumn("prediction", F.lit(linea_base_ingenua))
rmse_base = ev_rmse.evaluate(pred_baseline)
print(f"Linea base ingenua (mediana) -> RMSE={rmse_base:.4f}\n")

resultados_jhan = {}
for nombre, pred in predicciones_jhan.items():
    rmse = ev_rmse.evaluate(pred)
    r2 = ev_r2.evaluate(pred)
    mae = ev_mae.evaluate(pred)
    reduccion_pct = (1 - rmse / rmse_base) * 100 if rmse_base else float("nan")
    resultados_jhan[nombre] = rmse
    print(f"{nombre:32s} RMSE={rmse:.4f}  R2={r2:.4f}  MAE={mae:.4f}  (reduccion vs base: {reduccion_pct:.1f}%)")

nombre_ganador = min(resultados_jhan, key=resultados_jhan.get)
print(f"\nModelo ganador (menor RMSE): {nombre_ganador}")

modelo_ganador = modelos_entrenados_jhan[nombre_ganador]
modelo_ganador.write().overwrite().save("/opt/artifacts/jhan/modelo_duracion")
print("Modelo ganador guardado en /opt/artifacts/jhan/modelo_duracion")

Linea base ingenua (mediana) -> RMSE=24653396.5808



LinearRegression base            RMSE=23136259.7375  R2=0.0054  MAE=4342154.1704  (reduccion vs base: 6.2%)


LinearRegression + regularizacion RMSE=23280744.8503  R2=-0.0070  MAE=4355295.4328  (reduccion vs base: 5.6%)


RandomForestRegressor            RMSE=7387473.0216  R2=0.8986  MAE=2332560.4385  (reduccion vs base: 70.0%)

Modelo ganador (menor RMSE): RandomForestRegressor


Modelo ganador guardado en /opt/artifacts/jhan/modelo_duracion


## Cierre de fases y alcance

Este notebook cubre las **5 fases de CRISP-DM hasta el modelado/evaluación**: Comprensión del
negocio → Comprensión de los datos → Preparación de los datos → Modelado → Evaluación.
**No incluye la Fase 6 (Despliegue):** poner el modelo a inferir sobre flujos en vivo es
contenido de Unidad 2 (Spark Structured Streaming + Kafka), declarado como dimensión U2 en
el brief.

## Hallazgo(s) de esta dimensión

Ejecutado de punta a punta contra `TRCU.csv` (397 354 flujos reales capturados por Suricata):

- **Calidad de datos:** tras filtrar `flow_duration > 0` y deduplicar por `flow_id`, solo
  quedan 39 728 de 397 354 flujos (~10%) — la mediana histórica completa de `flow_duration`
  es 0, es decir, más de la mitad de los flujos capturados son instantáneos (un solo paquete
  o duración no medible). La línea base ingenua queda en RMSE=24.65M por esa razón.
- **Modelado:** `RandomForestRegressor` reduce el RMSE en **70.0%** frente a la línea base
  (7.39M vs. 24.65M) con **R² = 0.8986**, superando ampliamente el criterio de éxito de la
  Fase 1 (≥20% de reducción). Los modelos lineales apenas se mueven (R² ≈ 0.005, y la versión
  regularizada queda incluso ligeramente peor en RMSE que la base sin regularizar) — la
  relación entre `flow_duration` y las features iniciales del flujo es marcadamente no lineal.
  `RandomForestRegressor` es el modelo guardado como ganador.
- **Por protocolo (flujos con duración > 0):** TCP tiene una duración mediana mucho menor
  (228 220 µs ≈ 0.23s, n=15 563) que UDP (9 211 691 µs ≈ 9.2s, n=87 203) — una señal directa
  para dimensionar ventanas de sesión distintas por protocolo en la dimensión U2 (Kafka).

## Cómo ejecutar este notebook (evidencia de contribución)

1. Levantar el laboratorio (`docker compose up -d` desde `pyspark/`).
2. El dataset real (`TRCU.csv`) ya está en `pyspark/data/`, montado en `/opt/data/` dentro del contenedor — no requiere ajustar `RUTA_DATOS`.
3. Ejecutar de punta a punta (`Run All`), sin intervención manual: el modelo ganador se elige y se guarda automáticamente en la Fase 5.
4. Confirmar la carpeta de salida (`!ls -R` o `os.walk`) sobre `/opt/artifacts/jhan/` (Parquet particionado + modelo guardado).
5. Capturar pantalla con reloj del sistema y usuario/perfil visibles.
6. Commit del notebook ejecutado al repositorio del equipo (`pyspark/artifacts/` no se versiona, ver `.gitignore`).